# ResNet34-UNet Fetal Head Ultrasound Segmentation Pipeline
This notebook provides a complete interactive workflow to train, validate, and run inference on ultrasound scans using a custom ResNet34-UNet architecture.

**Objective**: Segment the fetal head region from 2D ultrasound scans, extract its boundary contour, and compute its circumference in pixels and physical units (mm/cm).

### Pipeline Components (Located in `src/`):
* `src/dataset.py`: Patient-aware train/val split, dataset class with automatic mask contour filling (scipy `binary_fill_holes`).
* `src/model.py`: Custom PyTorch ResNet34-UNet utilizing pretrained encoder weights.
* `src/loss.py`: Joint BCE + Dice loss module.
* `src/train.py`: Training CLI tool supporting mixed-precision (AMP) training, validation, early stopping, and automatic curve plotting.
* `src/inference.py`: Model loader, connected component post-processing, contour boundary extraction, exact perimeter calculator, and visualization.

## 1. Setup Environment & Imports

If running on **Google Colab**, you must first mount your Google Drive and navigate to the directory containing this project so that Colab can find `requirements.txt` and the `src` folder. Un-comment and run the mount cells below.

In [ ]:
# 1. Un-comment to Mount Google Drive (if using Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# 2. Un-comment and change path to your specific Google Drive project directory
# %cd /content/drive/MyDrive/Task - Segmentation

# 3. Install required dependencies
!pip install -r requirements.txt

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure CUDA is available (highly recommended for Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("PyTorch version:", torch.__version__)

# Check dataset count
if os.path.exists("images") and os.path.exists("masks"):
    print(f"Found {len(os.listdir('images'))} ultrasound scans and {len(os.listdir('masks'))} masks.")
else:
    print("WARNING: Please verify that 'images' and 'masks' directories are present in the current workspace.")

## 2. Preprocessing & Mask Filling Visualization
The raw masks of this dataset represent only the **boundary contour (ellipse line)** of the fetal head. Training a UNet directly on thin boundaries suffers from extreme class imbalance.

Our pipeline automatically applies `scipy.ndimage.binary_fill_holes` to fill the interior of the contour, converting it into a solid binary mask (filled ellipse) before feeding it to the model. Let's visualize this.

In [ ]:
from dataset import UltrasoundDataset

# Locate sample files
sample_image = "images/000_HC.png"
sample_mask = "masks/000_HC_Annotation.png"

if os.path.exists(sample_image) and os.path.exists(sample_mask):
    # Load dataset with target resize size (256, 256) for training
    ds = UltrasoundDataset([sample_image], [sample_mask], img_size=(256, 256))
    img_tensor, mask_tensor = ds[0]

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Raw Image
    axes[0].imshow(Image.open(sample_image), cmap="gray")
    axes[0].set_title("Original Ultrasound Scan")
    axes[0].axis("off")

    # Raw Mask (Contour only)
    axes[1].imshow(Image.open(sample_mask), cmap="gray")
    axes[1].set_title("Raw Mask (Boundary Outline)")
    axes[1].axis("off")

    # Filled Mask (Preprocessed)
    axes[2].imshow(mask_tensor.squeeze().numpy(), cmap="gray")
    axes[2].set_title("Preprocessed Mask (Filled Ellipse)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Could not find sample files to visualize. Please verify paths.")

## 3. Training the ResNet34-UNet
We use the production-ready script `src/train.py` via command line. It uses a **patient-aware train-validation split** to avoid patient-level data leakage, uses ImageNet pretrained encoder weights, and enables Automatic Mixed Precision (AMP) to speed up training on Colab GPU.

*Adjust the parameters below as needed. We recommend training for 15-30 epochs to reach high accuracy.*

In [ ]:
# Run training. Best model weights and history will be saved in 'checkpoints/'
!python src/train.py \
    --images_dir "images" \
    --masks_dir "masks" \
    --save_dir "checkpoints" \
    --epochs 20 \
    --batch_size 16 \
    --lr 2e-4 \
    --img_size 256 \
    --val_size 0.2 \
    --patience 7 \
    --num_workers 2 \
    --pretrained True

## 4. Plot Training & Validation History
Let's display the learning curves automatically generated by the training script.

In [ ]:
curves_path = "checkpoints/training_curves.png"
if os.path.exists(curves_path):
    plt.figure(figsize=(10, 10))
    img_curves = Image.open(curves_path)
    plt.imshow(img_curves)
    plt.axis("off")
    plt.show()
else:
    print("Training curves plot not found. Run the training script first.")

## 5. Inference & Circumference (Perimeter) Calculation
We load the trained model, run inference, post-process predictions by keeping the largest connected component, extract the outer contour boundary, and calculate the perimeter in pixels and mm (assuming `0.1 mm/pixel` spacing fallback).

In [ ]:
from inference import load_model, run_inference, visualize_results

model_checkpoint = "checkpoints/best_model.pth"

if os.path.exists(model_checkpoint):
    # Load model to GPU/CPU
    model = load_model(model_checkpoint, device)

    # Choose a sample from the images folder
    test_image_path = "images/001_HC.png"

    # Run inference pipeline (resizes mask back to original resolution for contour extraction)
    # Set pixel_spacing to your physical units calibration (e.g. 0.1 mm/pixel)
    results = run_inference(
        model=model,
        image_path=test_image_path,
        img_size=(256, 256),
        threshold=0.5,
        pixel_spacing=0.1,
        device=device
    )

    # Print perimeter calculations
    print(f"--- Calculations for {os.path.basename(test_image_path)} ---")
    print(f"Head Circumference (Pixels): {results['perimeter_pixels']:.2f} px")
    if results['perimeter_mm'] > 0:
        print(f"Head Circumference (Real World): {results['perimeter_mm']:.2f} mm")
        print(f"Head Circumference (Real World): {results['perimeter_mm']/10.0:.2f} cm")

    # Visualize results
    visualize_results(results, test_image_path, save_path="inference_output.png")
else:
    print("No model checkpoint found. Please train the model or provide a trained checkpoint first.")